In [ ]:
from cellpose import models, io
import pyclesperanto as cle
from bioio import BioImage
import skimage as sk
from skimage.segmentation import relabel_sequential, clear_border
from skimage.morphology import remove_small_objects, ball
from skimage.transform import rescale, resize
from skimage.io import imsave, imread
from skimage.measure import label, regionprops, regionprops_table, marching_cubes, mesh_surface_area, find_contours
from scipy import ndimage, spatial
import numpy as np
from glob import glob
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import scipy as sc
import statsmodels.api as sm
import statsmodels.formula.api as smf
from tqdm import tqdm
from joblib import Parallel, delayed
import re
from pathlib import Path
from random import sample
import napari

cle.select_device("NVIDIA")

In [ ]:
def read_img(file):
    return np.squeeze(BioImage(file).get_image_data("ZYX", C=1))

def get_pixel_spacing(file):
    ''' 
    Use one image from dataset read in with BioImage without .get_image_data to access pixel spacing
    '''
    img = BioImage(file)
    pixel_spacing = (img.physical_pixel_sizes[1],img.physical_pixel_sizes[2])
    voxel = (img.physical_pixel_sizes[0],img.physical_pixel_sizes[1],img.physical_pixel_sizes[2])
    return pixel_spacing, voxel

def normalize_images(input_image,save_path,name,tophat_radius=10):
    input_gpu = cle.push(np.squeeze(input_image))
    #normalizing the image stack
    equalized_intensities_stack = cle.create_like(input_gpu)
    a_slice = cle.create([input_gpu.shape[1], input_gpu.shape[2]])
    num_slices = input_gpu.shape[0]
    mean_intensity_stack = cle.mean_of_all_pixels(input_gpu)
    corrected_slice = None
    for z in range(0, num_slices):
        # get a single slice out of the stack
        cle.copy_slice(input_gpu, a_slice, z)
        # measure its intensity
        mean_intensity_slice = cle.mean_of_all_pixels(a_slice)
        # correct the intensity
        correction_factor = mean_intensity_slice/mean_intensity_stack
        corrected_slice = cle.multiply_image_and_scalar(a_slice, corrected_slice, correction_factor)
        # copy slice back in a stack
        cle.copy_slice(corrected_slice, equalized_intensities_stack, z)
    #background subtraction (increase the signal to noise ratio for improved segmentation results)
    background_subtracted_top_hat = cle.top_hat_sphere(equalized_intensities_stack,radius_x=tophat_radius,radius_y=tophat_radius,radius_z=tophat_radius)
    #filtered = cle.subtract_images(white_tophat,background_subtracted_top_hat)
    #pull data off gpu
    #cle.pull(filtered)
    output_cpu = np.squeeze(cle.pull(background_subtracted_top_hat))
    #footprint = ball(6)
    #output = white_tophat(output_cpu,footprint)
    #imsave(os.path.join(save_path,f'{name}_Ch2_processed_tophat_10.tif'),output_cpu)
    return output_cpu

def resample_stack(img,z=0.5):
    input_gpu = cle.push(img)
    reduced_stack = cle.scale(input_gpu,factor_x=1.0,factor_y=1.0,factor_z=z,resize=True,interpolate=True)
    new_stack = cle.pull(reduced_stack)
    return new_stack


def sato_filter(img):
    input_gpu = cle.push(np.squeeze(img))
    a_slice = cle.create([input_gpu.shape[1], input_gpu.shape[2]])
    num_slices = input_gpu.shape[0]
    sato_filtered_stack = cle.create_like(input_gpu)
    for z in range(0, num_slices):
        # get a single slice out of the stack
        cle.copy_slice(input_gpu, a_slice, z)
        # apply sato filter
        gaus_filtered = cle.gaussian_blur(a_slice,sigma_x=1.0,sigma_y=1.0)
        sato_filtered_z = cle.sato_filter(gaus_filtered,sigma_minimum=1,sigma_maximum=8,sigma_step=2)
        #copy back into stack
        cle.copy_slice(sato_filtered_z, sato_filtered_stack, z)
    cle.pull(input_gpu)
    return cle.pull(sato_filtered_stack)

def down_size(img, scale=None):
   scale == 1 if scale == None else scale == scale
   if scale == None:
       print('no scaling provided, using scaling of 1 (no scaling applied)')
   return rescale(img, scale, anti_aliasing=True, channel_axis=None,preserve_range=True)

def get_middle_slices(img, num_slices=4):
    if img.shape[0] < num_slices:
        print(f'Image has {img.shape[0]} slices, which is less than the requested {num_slices} slices. Returning all slices.')
        return img
    else:
        middle_index = img.shape[0] // 2
        half_num_slices = num_slices // 2
        start_index = max(0, middle_index - half_num_slices)
        end_index = min(img.shape[0], middle_index + half_num_slices)
        return list(np.unstack(img[start_index:end_index]))

def unstack(img):
    return list(np.unstack(img))

def run_cp(imgs):
    '''
    Give function list of all resized images for more efficient processing
    '''
    #io.logger_setup()
    model = models.CellposeModel(gpu=True, pretrained_model="cpsam_v2")
    masks, _, _ = model.eval(imgs, batch_size=32, diameter=90,resample= True, channel_axis=None,min_size=200,normalize=True,cellprob_threshold=0.0)
    return masks

def up_size_masks(masks, img):
    if masks.shape == img.shape:
        print('masks and image are the same size, no resizing applied')
        return masks
    else:
        return resize(masks,img.shape,preserve_range=True,order=0).astype(np.uint16)

def clear_xy_borders(mask_img,border):
    bool_arr = np.zeros((mask_img.shape[0],mask_img.shape[1]-border*2,mask_img.shape[2]-border*2))
    xy_boarder = np.pad(bool_arr,pad_width=((0,0),(border,border),(border,border)),mode='constant',constant_values=1).astype(np.bool)
    labels, number = label(mask_img,background=0,return_num=True)
    indices = np.arange(number+1)
    border_indices = np.unique(labels[xy_boarder])
    label_mask = np.isin(indices, border_indices)
    mask = label_mask[labels.reshape(-1)].reshape(labels.shape)
    labels[mask] = 0
    return labels
     

def process_masks(mask_img, save_path, name):
    cleared_borders = [clear_border(mask,buffer_size=6) for mask in mask_img]
    rmv_small_objs = [remove_small_objects(mask,max_size=200) for mask in cleared_borders]
    seq_lables = [relabel_sequential(mask)[0] for mask in rmv_small_objs]
    stack = np.stack(seq_lables,axis=0)
    imsave(os.path.join(save_path,f'{name}_summed_chs_labels_cleared_borders.tif'),stack,check_contrast=False)
    return seq_lables

def _label_sa_solidity_pre(label_val, submask, pixel_spacing):
    m = submask.astype(np.uint8)

    # surface area
    padded = np.pad(m, 1, mode='constant')
    verts, faces, _, _ = marching_cubes(padded, level=0.5, spacing=pixel_spacing)
    surface_area = mesh_surface_area(verts, faces)

    # solidity on the same small crop — convex hull is cheap here
    sub_props = regionprops(m)[0]
    solidity = sub_props.solidity

    return label_val, surface_area, solidity


def median_deviation_from_convex_hull(regionmask):
    contour_pts = np.round(find_contours(regionmask)[0]).astype(np.int16)
    convex_hull = spatial.ConvexHull(contour_pts).simplices
    convex_hull_pts = np.vstack(contour_pts[convex_hull])
    print(convex_hull_pts)
    contour_tree = spatial.KDTree(contour_pts)
    convex_tree = spatial.KDTree(convex_hull_pts)
    distances = convex_tree.sparse_distance_matrix(contour_tree,max_distance=10,output_type='dok_array')
    distances_list = distances.toarray()
    filtered_dists = distances_list[np.nonzero(distances_list)]
    print(filtered_dists)
    median_distance = np.median(filtered_dists)
    return median_distance

def measure_objects(mask_img, pixel_spacing, name, batch):
    # 1. Standard descriptors — one fast pass.
    #    Pass spacing so 'area' is volume in physical units (e.g. µm³).
    #print('collecting regions properties')
    df_slice = []
    for z in mask_img:
        if np.max(z) >= 1:
            print(np.max(z))
            d = pd.DataFrame(regionprops_table(z, spacing=pixel_spacing, 
                                               properties=['label', 'area', 'centroid', 'axis_major_length', 'axis_minor_length','equivalent_diameter_area', 'extent',  'solidity', 'eccentricity', 'perimeter_crofton']))
            df_slice.append(d)
    df = pd.concat(df_slice,ignore_index=True)
    # 2. Pre-crop each object in the main process so workers receive only
    #    a tiny bool array, not the full label volume.
    #print('finding objects with ndimage')
    #objects = ndimage.find_objects(mask_img)
    #print(f'extracting subregions for {len(objects)}')
    #tasks = []
    #for idx, sl in enumerate(objects):
    #    if sl is None:
    #        continue
    #    label_val = idx + 1
    #    padded_sl = tuple(slice(max(0, s.start - 1), s.stop + 1) for s in sl)
    #    submask = (mask_img[padded_sl] == label_val)   # small bool array
    #    tasks.append((label_val, submask))

    # 3. Parallelize only the costly surface-area extraction.
    #print('Extracting suface areas for objects')
    #results = Parallel(n_jobs=n_jobs)(
    #    delayed(_label_sa_solidity_pre)(label_val, submask, pixel_spacing)
    #    for label_val, submask in tasks
    #)
    #sa_df = pd.DataFrame(results, columns=['label', 'surface_area', 'solidity'])
    # 4. Merge on label — order-independent, gap-safe.
    #print(props_df.columns)
    #df = props_df.merge(sa_df, on='label', how='left')
    # 5. Sphericity as a vectorized op, now that V and A are aligned columns.
    #    'area' is volume because spacing was passed to regionprops.
    df['roundness'] = (4*np.pi*df['area'])/ df['perimeter_crofton']**2
    df['SA_to_A_ratio'] = df['perimeter_crofton']/df['area']
    df['image'] = name
    df['batch'] = batch
    condition = 'WT_ctrl' if 'WT' in name else 'DKO_ctrl'
    df['condition'] = condition
    fish_id = name.split('_')[2]
    df['fish_id'] = f'{fish_id}_{condition}_{batch}'
    return df


def merge_dfs_save(dfs, save_loc):
    all_data = pd.concat(dfs,ignore_index=True)
    all_data.to_csv(os.path.join(save_loc,'Zebrafish_desmosome_df_all_data_2d_analysis.csv'),index=False)

In [ ]:
io.logger_setup()

In [ ]:
# full run
path = Path(r"L:\lsdcc\biu-users\Ikonen-group\XinZhou\20260625_zebrafish\fin")
#save_img_path = Path(r"L:\lsdcc\biu-users\Ikonen-group\XinZhou\20260625_zebrafish\Cell_Masks_2D\Summed_Ch")
save_processed_path = Path(r"L:\lsdcc\biu-users\Ikonen-group\XinZhou\20260625_zebrafish\Cell_Masks_2D\Processed_Ch2\tophat")
#os.makedirs(os.path.join(save_img_path,'Batch_1'),exist_ok=True)
#os.makedirs(os.path.join(save_img_path,'Batch_2'),exist_ok=True)
os.makedirs(os.path.join(save_processed_path,'Batch_1'),exist_ok=True)
os.makedirs(os.path.join(save_processed_path,'Batch_2'),exist_ok=True)
#save_df_path = Path(r"L:\lsdcc\biu-users\Ikonen-group\XinZhou\20260625_zebrafish")
#scale = [0.5,0.2,0.2]
files = sample(sorted(glob(f"{path}/*/*tail*.czi")),5)
basenames = list(map(os.path.basename,files))
names = [name[0:-4] for name in basenames]
batches = [re.search(r"Batch_\d{1}",file).group() for file in files]
#pixel_spacing = get_pixel_spacing(files[0])
#dfs = []

    #stack = unstack(norm_img)
    #masks = run_cp(stack)
    #cleared_borders = process_masks(masks, save_path_masks, name)
    #df = measure_objects(cleared_borders,pixel_spacing,name,batch)
    #dfs.append(df)
#merge_dfs_save(dfs,save_df_path)

In [ ]:
imgs = [read_img(file) for file in files]
norm_imgs = [normalize_images(img,tophat_radius=5,save_path=None,name=None) for img in imgs]

In [ ]:
pixel_spacing, voxel = get_pixel_spacing(files[0])

In [ ]:
anisotropy = voxel[0]/voxel[1]

In [ ]:
anisotropy

In [ ]:
model = models.CellposeModel(gpu=True, pretrained_model="cpsam_v2")
masks, _, _ = model.eval(norm_imgs[0], batch_size=32, diameter=90, resample= True, do_3D=True, z_axis=0, channel_axis=None, min_size=200,normalize=True,cellprob_threshold=0.0,anisotropy=anisotropy)

In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(norm_imgs[0])
v.add_labels(masks)

In [ ]:
for f in files:
    print(get_pixel_spacing(f))

In [ ]:
reduced_imgs = [resample_stack(img) for img in tqdm(norm_imgs)]
sato_filter_imgs = [sato_filter(img) for img in tqdm(reduced_imgs)]

In [ ]:
for img in reduced_imgs:
    v.add_image(img, name='resampled')

In [ ]:
meijering_filter = [sk.filters.meijering(img,black_ridges=False) for img in tqdm(norm_imgs)]
sato_filters = [sk.filters.sato(img, black_ridges=False) for img in tqdm(norm_imgs)]

In [ ]:
#Testing filters to improve membrane outlines
test_img = read_img(files[0])
#input_gpu = cle.push(test_img)


In [ ]:
test_img = sk.img_as_float(test_img)
stack = unstack(test_img)

In [ ]:
test_slice = stack[25]
seed = np.copy(test_slice)

In [ ]:
sk.filters.try_all_threshold(test_slice,figsize=(11,7))

In [ ]:
yen_mask = test_slice > sk.filters.threshold_local(test_slice)
plt.imshow(test_slice,cmap='gray')
plt.imshow(yen_mask,cmap='Greens_r',alpha=0.5)

In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(test_slice)
v.add_labels(yen_mask)

In [ ]:
#reprocess masks to clear borders correctly and then recollect measurements
files_dir = Path(r"L:\lsdcc\biu-users\Ikonen-group\XinZhou\20260625_zebrafish\Cell_Masks_2D")
save_df_path = Path(r"L:\lsdcc\biu-users\Ikonen-group\XinZhou\20260625_zebrafish")
files = sorted(glob(f"{files_dir}/*/*.tif"))
basenames = list(map(os.path.basename,files))
names = [name[0:-4] for name in basenames]
batches = [re.search(r"Batch_\d{1}",file).group() for file in files]
dfs = []
for file, name, batch in tqdm(zip(files,names,batches)):
    save_path = os.path.join(files_dir,batch)
    label_img = imread(file)
    unstack = list(np.unstack(label_img))
    cleared_borders = process_masks(unstack,save_path,name)
    #df = measure_objects(cleared_borders,pixel_spacing,name,batch)
    #dfs.append(df)
merge_dfs_save(dfs, save_df_path)

In [ ]:
print(files)

In [ ]:
masks_path = r"E:\KristinG\20260625_zebrafish\fin_outputs\Cell_Masks"
save_new_masks_path = r"E:\KristinG\20260625_zebrafish\fin_outputs\Filtered_Cell_Masks"
save_df_path = r"E:\KristinG\20260625_zebrafish\fin_outputs"
pixel_spacing = get_pixel_spacing(BioImage(r"E:\KristinG\20260625_zebrafish\fin\DKO_ctrl_1_skin_1_ventral_fin_Out.czi"))
mask_files = sorted(glob(rf'{masks_path}\*.tif'))
masks = list(map(imread,mask_files))

# Plotting Data and Testing

In [ ]:
import pandas.plotting as pdplt

In [ ]:
batch1 = pd.read_csv('/home/gkristin/Zebrafish_desmosomes/Zebrafish_desmosome_tail_data.tsv',sep='\t')
batch2 = pd.read_csv('/home/gkristin/Zebrafish_desmosomes/Zebrafish_desmosome_df_2nd_batch.tsv',sep='\t')

In [ ]:
batch1['batch'] = 1

In [ ]:
batch1.rename(columns={'groups':'group','regions':'region'},inplace=True)

In [ ]:
batch2.columns

In [ ]:
merged_df = pd.concat([batch1,batch2],ignore_index=True)

In [ ]:
merged_df.to_csv('/home/gkristin/Zebrafish_desmosomes/Zebrafish_desmosome_df_tail_all_batches.tsv',sep='\t',index=False)

## Stats testing

Guided by Biostatistics group near BIU

In [ ]:
df = pd.read_csv(Path(r"L:\lsdcc\biu-users\Ikonen-group\XinZhou\20260625_zebrafish\Zebrafish_desmosome_df_all_data_2d_analysis.csv"))

In [ ]:
df['']

In [ ]:
fish_median = df.groupby(['batch','group','fish_id'],as_index=False).median(numeric_only=True)

In [ ]:
fish_median.tail()

### Welch's T-test

In [ ]:
batch1_DKO = fish_median.query('batch == 1 & group == "DKO-ctrl"')
batch2_DKO = fish_median.query('batch == 2 & group == "DKO-ctrl"')
batch1_WT = fish_median.query('batch == 1 & group == "WT-ctrl"')
batch2_WT = fish_median.query('batch == 2 & group == "WT-ctrl"')

wt = fish_median.query('group == "WT-ctrl"')
dko = fish_median.query('group == "DKO-ctrl"')


In [ ]:
batch1_WT['surface_area'].to_numpy()

In [ ]:
#testing batch differences:
wt_sa_batches = sc.stats.ttest_ind(batch1_WT['surface_area'].to_numpy(),batch2_WT['surface_area'].to_numpy(),equal_var=False)
wt_sp_batches = sc.stats.ttest_ind(batch1_WT['sphericity'].to_numpy(),batch2_WT['sphericity'].to_numpy(),equal_var=False)
dko_sa_batches = sc.stats.ttest_ind(batch1_DKO['surface_area'].to_numpy(),batch2_DKO['surface_area'].to_numpy(),equal_var=False)
dko_sp_batches = sc.stats.ttest_ind(batch1_DKO['sphericity'].to_numpy(),batch2_DKO['sphericity'].to_numpy(),equal_var=False)


In [ ]:
print(f"""
    p-values from the following tests:
    WT batches S.A. Welch's T-test: {wt_sa_batches.pvalue}
    WT batches Sphericity Welch's T-test: {wt_sp_batches.pvalue}
    DKO batches S.A. Welch's T-test: {dko_sa_batches.pvalue}
    DKO batches Sphericity Welch's T-test: {dko_sp_batches.pvalue}
"""
    )

In [ ]:
wt_v_dko_sa = sc.stats.ttest_ind(wt['surface_area'].to_numpy(),dko['surface_area'].to_numpy(),equal_var=False)
st_v_dko_sphericity = sc.stats.ttest_ind(wt['sphericity'].to_numpy(),dko['sphericity'].to_numpy(),equal_var=False)
print(f"""
    p-values from the following tests:
    WT v DKO S.A. Welch's T-test: {wt_v_dko_sa.pvalue}
    WT v DKO Sphericity Welch's T-test: {st_v_dko_sphericity.pvalue}
"""
    )

### Ordinary least squares linear model

In [ ]:
fish_median['group'].unique()

In [ ]:
print(fish_median['surface_area'])

In [ ]:
sphericity_mod = smf.ols(formula='sphericity ~ group + batch',data=fish_median)
sa_mod = smf.ols(formula='surface_area ~ group + batch',data=fish_median)

In [ ]:
sphericity_res = sphericity_mod.fit()
sa_res = sa_mod.fit()

In [ ]:
dir(sphericity_res)

In [ ]:
# Source - https://stackoverflow.com/a/51734632
# Posted by Michael B
# Retrieved 2026-07-21, License - CC BY-SA 4.0

def results_summary_to_dataframe(results):
    '''take the result of an statsmodel results table and transforms it into a dataframe'''
    pvals = results.pvalues
    coeff = results.params
    conf_lower = results.conf_int()[0]
    conf_higher = results.conf_int()[1]

    results_df = pd.DataFrame({"pvals":pvals,
                               "coeff":coeff,
                               "conf_lower":conf_lower,
                               "conf_higher":conf_higher
                                })

    #Reordering...
    results_df = results_df[["coeff","pvals","conf_lower","conf_higher"]]
    return results_df


In [ ]:
print(f"""
{sa_res.summary2()} \n
======================================================= \n
Confidence Interval \n
{sa_res.conf_int(alpha=.05)})
""")

In [ ]:
print(sa_res.summary(),sa_res.conf_int(alpha=0.05))

## Making Plots

In [ ]:
# create plots for easy vis
fig, axs = plt.subplots(2,3,figsize=(10,8))
sns.violinplot(data=fish_median, x='group', y='area', hue='batch', split='batch',gap=0.15,fill=False, hue_order=['1','2'],inner='stick',ax=axs[0,0])
axs[0,0].set_title('Vol')
axs[0,0].set_ylabel('Vol (um³)')
sns.violinplot(data=fish_median, x='group', y='equivalent_diameter_area',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['1','2'],inner='stick',ax=axs[0,1])
axs[0,1].set_title('Equivalent Diameter Vol')
axs[0,1].set_ylabel('Vol (um³)')
sns.violinplot(data=fish_median, x='group', y='extent',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['1','2'],inner='stick',ax=axs[0,2])
axs[0,2].set_title('Extent')
axs[0,2].set_ylabel('Extent [0-1]')
sns.violinplot(data=fish_median, x='group', y='surface_area',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['1','2'],inner='stick',ax=axs[1,0])
axs[1,0].set_title('Surface Area')
axs[1,0].set_ylabel('Area (um²)')
sns.violinplot(data=fish_median, x='group', y='solidity',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['1','2'],inner='stick',ax=axs[1,1])
axs[1,1].set_title('Solidity')
axs[1,1].set_ylabel('Solidity [0-1]')
sns.violinplot(data=fish_median, x='group', y='sphericity',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['1','2'],inner='stick',ax=axs[1,2])
axs[1,2].set_title('Sphericity')
axs[1,2].set_ylabel('Sphericity [0-1]')
plt.tight_layout()
plt.savefig('/home/gkristin/Zebrafish_desmosomes/Cell_Shape_Features_Plot_Batches_fish_medians.png',dpi=300)
plt.show()

In [ ]:
pg = sns.PairGrid(fish_median,hue='batch',vars=['area','equivalent_diameter_area', 'extent', 'surface_area', 'solidity', 'sphericity'])
pg.map_upper(sns.scatterplot,style=fish_median['group'])
pg.map_diag(sns.histplot,)
pg.add_legend()
#pg.savefig('/home/gkristin/Zebrafish_desmosomes/Cell_Shape_Features_Plot_Batches_fish_medians.png',dpi=300)

Plots to make:
Use the Vol, equivalent, SA, and Sphericity just for tail as individual plots
Some relationship plotting of the above measurements

In [ ]:
df = pd.read_table('/home/gkristin/Zebrafish_desmosomes/Zebrafish_desmosome_df_added_filter_columns.tsv',sep='\t')

In [ ]:
g = sns.catplot(data=df_tail, x='groups', y='equivalent_diameter_area', hue='groups',kind='violin',fill=False, order=['WT-ctrl','DKO-ctrl'],inner='quart')
g.set_axis_labels('Groups','Vol (um³)')
g.set_titles(col_template='Volume per cell')

In [ ]:
fig, axs = plt.subplots(1,1,figsize=(5,5))
sns.violinplot(data=df_tail, x='groups', y='sphericity', hue='groups', fill=False, order=['WT-ctrl','DKO-ctrl'],inner='quart',ax=axs)
axs.set_title('Sphericity per Cell')
axs.set_ylabel('Sphericity [0-1]')
axs.set_ylim(0,1)
plt.tight_layout()
plt.savefig('/home/gkristin/Zebrafish_desmosomes/Sphericity_Plot.png',dpi=300)

In [ ]:
from itertools import combinations

In [ ]:
grouped_df = (df.groupby('Image').median(numeric_only=True))

In [ ]:
def assign_condition(r):
    if 'WT' in r:
        return 'WT'
    elif 'DKO' in r:
        return 'DKO'
grouped_df['groups'] = [assign_condition(r) for r in grouped_df.index]

In [ ]:
def assign_condition(r):
    if 'tail' in r:
        return 'tail'
    elif 'dorsal' in r:
        return 'dorsal'
    elif 'ventral' in r:
        return 'ventral'
grouped_df['regions'] = [assign_condition(r) for r in grouped_df.index]

In [ ]:
measurements = ['area','equivalent_diameter_area','surface_area','sphericity']

In [ ]:
combos = list(combinations(measurements,2))

In [ ]:
pg = sns.PairGrid(fish_median.query("group == 'DKO-ctrl'"),hue='batch',diag_sharey=False,vars=measurements,palette=sns.color_palette(palette='colorblind',n_colors=2))
pg.map_lower(sns.scatterplot)
pg.map_diag(sns.histplot,bins=10)
pg.map_upper(sns.regplot,scatter=True,logx=True)
pg.add_legend(title='Batch ID',adjust_subtitles=True)
pg.savefig('/home/gkristin/Zebrafish_desmosomes/Paired_Grid_Plot_DKO_Batches_Fish_Median.png',dpi=300)


In [ ]:
sns.lmplot(df.query("regions == 'tail'"),x=combos[1][0],y=combos[0][1],hue='groups')

In [ ]:
df_tail = grouped_df.query("regions == 'tail'")

In [ ]:
df_tail.value_counts('groups')

In [ ]:
def assign_fish_id(r):
    values = r.split('_')
    fish_id = values[2]
    return fish_id
grouped_df['fish_id'] = [assign_fish_id(r) for r in grouped_df.index]


In [ ]:
grouped_df

In [ ]:
df_tail = grouped_df.query("regions == 'tail'")

In [ ]:
df_tail.value_counts(['groups'])